# ExoJury 🪐⚖️ — every Kepler candidate gets a fair trial

**India High School Exoplanet Data Challenge (Celesta)**

This notebook walks through the whole project on the NASA KOI cumulative
table: the leakage audit, the honest model, calibration, conformal
prediction, the label audit that found real catalog errors, and the ranked
candidate frontier. It runs from saved pipeline artifacts in seconds; to
retrain from scratch, see the README.


In [1]:
import sys, joblib
import numpy as np, pandas as pd
sys.path.insert(0, "../src")
from exojury import config, data
from exojury.calibrate import conformal_sets, reliability_table, expected_calibration_error

df = data.load_raw()
print(df.shape)
df["koi_disposition"].value_counts()

(9564, 140)


koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2747
CANDIDATE         1978
Name: count, dtype: int64

## 1. The dataset contains the answer key

Before modeling, we checked every one of the 140 columns for **leakage** —
columns that are *outputs* of NASA's vetting process rather than
observations. Four kinds hide here: `kepler_name` (only confirmed planets
get names), `koi_pdisposition` (the pipeline's own verdict),
`koi_comment` (the vetting rationale in text), and the Robovetter
`koi_fpflag_*` flags.

In [2]:
# The "answer key" classifier: two lines, zero ML
pred = np.where(df["kepler_name"].notna(), "CONFIRMED",
        np.where(df["koi_pdisposition"] == "FALSE POSITIVE", "FALSE POSITIVE", "CANDIDATE"))
print(f"Answer-key accuracy: {(pred == df['koi_disposition']).mean():.4f}")

anyflag = df[config.VETTING_FLAG_COLS].sum(axis=1) > 0
lab = df[df["koi_disposition"] != "CANDIDATE"]
pred2 = np.where(anyflag[lab.index], "FALSE POSITIVE", "CONFIRMED")
print(f"fpflags-only accuracy (binary): {(pred2 == lab['koi_disposition']).mean():.4f}")

Answer-key accuracy: 0.9996
fpflags-only accuracy (binary): 0.9851


Any model trained with these columns is partly copying the
teacher's answer sheet. Our leakage policy (`src/exojury/config.py`)
excludes them all. **Everything below uses physics only.**

![leakage](../reports/figures/04_leakage_answer_key.png)

## 2. Exploration

![classes](../reports/figures/01_class_balance.png)
![period-radius](../reports/figures/02_period_radius.png)

False positives dominate the "giant planet" regime — objects with fitted
radii above ~2 Jupiter radii are almost always eclipsing binary stars, not
planets. Feature separations follow known astrophysics (centroid offsets ⇒
background binaries; odd-even depth differences ⇒ eclipsing binaries):

![separation](../reports/figures/03_feature_separation.png)

## 3. The honest model

Binary task: CONFIRMED vs FALSE POSITIVE (the CANDIDATE rows are *unlabeled
by definition* — we score them in §6 instead of pretending they're a
class). Gradient boosting (`HistGradientBoostingClassifier`), stratified
80/20 split, 105 physics features, NaNs handled natively.

In [3]:
raw = joblib.load(config.MODELS_DIR / "baseline_honest.joblib")
test_df = pd.read_csv(config.MODELS_DIR / "test_split.csv")
y_test = test_df["koi_disposition"].map(config.LABEL_MAP_BINARY).values
X_test = data.build_features(test_df)[raw["features"]]

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
p = raw["model"].predict_proba(X_test)[:, 1]
print(f"Held-out test: ROC-AUC={roc_auc_score(y_test, p):.4f}  "
      f"acc={accuracy_score(y_test, p >= .5):.4f}  f1={f1_score(y_test, p >= .5):.4f}")

Held-out test: ROC-AUC=0.9967  acc=0.9822  f1=0.9754


Including the Robovetter flags would add +1.3 accuracy
points — inflation we *measured* rather than shipped (see README table).
For the challenge's 3-class formulation we get 86.1% accuracy / 0.831
macro-F1 (`python -m exojury.three_class`); CANDIDATE is the weak class
(F1 0.68) precisely because it's a state of knowledge, not a kind of object.

![confusion](../reports/figures/05_confusion_3class.png)

## 4. Honest uncertainty: calibration + a 95% guarantee

A probability is only useful if it's *calibrated* — "90%" should come true
~90% of the time. We use isotonic calibration, then **split conformal
prediction** on a held-aside calibration slice: every object gets a
prediction *set* guaranteed to contain the truth 95% of the time. An empty
set means "typical of neither class" → **NEEDS REVIEW**.

In [4]:
cal = joblib.load(config.MODELS_DIR / "calibrated_conformal.joblib")
p_cal = cal["model"].predict_proba(data.build_features(test_df)[cal["features"]])[:, 1]
print(f"ECE after calibration: {expected_calibration_error(y_test, p_cal):.4f}")
sets_test = conformal_sets(p_cal, cal["qhat"])
covered = np.where(y_test == 1,
                   sets_test.isin(["CONFIRMED", "NEEDS REVIEW (both plausible)"]),
                   sets_test.isin(["FALSE POSITIVE", "NEEDS REVIEW (both plausible)"]))
print(f"Empirical coverage: {covered.mean():.4f} (target 0.95, qhat={cal['qhat']:.3f})")
sets_test.value_counts()

ECE after calibration: 0.0090
Empirical coverage: 0.9559 (target 0.95, qhat=0.189)


FALSE POSITIVE              948
CONFIRMED                   519
NEEDS REVIEW (ambiguous)     51
Name: count, dtype: int64

![reliability](../reports/figures/06_reliability.png)
![conformal](../reports/figures/07_conformal_verdicts.png)

## 5. Auditing NASA's own labels 🏆

Confident learning (cleanlab) asks: which catalog labels does an
out-of-fold model most confidently disagree with? We checked the top flags
against the literature **after** the model produced them:

| KOI | Catalog (our snapshot) | Model p(planet) | Literature |
|---|---|---|---|
| K01450.01 (Kepler-854 b) | CONFIRMED | 0.00003 | **Demoted to false positive** (Niraula et al. 2022) |
| K01416.01 (Kepler-840 b) | CONFIRMED | 0.0016 | **Demoted to false positive** (same study) |
| K07016.01 (Kepler-452 b) | CONFIRMED | 0.0006 | Validation formally disputed (Mullally et al. 2018) |
| K03794.01 (Kepler-1520 b) | CONFIRMED | 0.003 | Real, but *disintegrating* — an honest miss |

The pipeline rediscovered real catalog corrections from this CSV alone.

In [5]:
audit = pd.read_csv(config.REPORTS_DIR / "label_audit.csv")
audit.head(10)

,kepoi_name,kepler_name,koi_disposition,koi_period,koi_prad,koi_model_snr,p_planet_oof
0,K01450.01,Kepler-854 b,CONFIRMED,2.144632,59.19,334.8,0.000031
1,K03278.01,NaN,FALSE POSITIVE,88.180468,2.78,17.5,0.999799
2,K02451.01,NaN,FALSE POSITIVE,13.374996,2.01,16.9,0.999644
3,K06233.01,Kepler-1645 b,CONFIRMED,16.178030,2.25,6.0,0.000435
4,K07016.01,Kepler-452 b,CONFIRMED,384.847556,1.09,12.3,0.000588
5,K03503.02,Kepler-1703 c,CONFIRMED,31.824530,0.80,7.7,0.000755
6,K02307.01,NaN,FALSE POSITIVE,4.150297,3.03,24.2,0.999168
7,K07368.01,Kepler-1974 b,CONFIRMED,6.842939,NaN,NaN,0.001016
8,K00245.04,Kepler-37 e,CONFIRMED,51.206903,NaN,NaN,0.001247
9,K01416.01,Kepler-840 b,CONFIRMED,2.495780,15.75,171.4,0.001601


## 6. The frontier: 1,978 unresolved candidates, ranked

Trained on labeled rows only, ExoJury scores every CANDIDATE with a
calibrated probability and conformal verdict — a telescope-time priority
list. 74% get a 95%-guaranteed verdict; the rest are honestly flagged for
human review.

In [6]:
scores = pd.read_csv(config.REPORTS_DIR / "candidate_scores.csv")
print(scores["conformal_verdict"].value_counts().to_string(), "\n")
scores.head(10)

conformal_verdict
FALSE POSITIVE              1125
NEEDS REVIEW (ambiguous)     518
CONFIRMED                    335 



,kepoi_name,koi_period,koi_prad,koi_teq,p_planet_calibrated,conformal_verdict
0,K01972.01,17.791087,2.88,830.0,1.0,CONFIRMED
1,K03394.01,15.576406,1.79,793.0,1.0,CONFIRMED
2,K00353.01,152.106043,11.58,458.0,1.0,CONFIRMED
3,K00298.02,57.384037,1.62,397.0,1.0,CONFIRMED
4,K02159.01,7.596700,1.32,964.0,1.0,CONFIRMED
5,K02656.01,6.773679,1.33,1083.0,1.0,CONFIRMED
6,K01145.01,30.587215,2.32,621.0,1.0,CONFIRMED
7,K03864.02,18.257272,0.94,547.0,1.0,CONFIRMED
8,K01573.02,7.136922,1.38,956.0,1.0,CONFIRMED
9,K01628.03,37.840423,1.76,607.0,1.0,CONFIRMED


## 7. AI vetting dossiers (Featherless.ai)

For any KOI, DeepSeek-V3 (via the sponsor's OpenAI-compatible API) writes
an astronomer-style report grounded strictly in the pipeline's numbers.
**The sklearn model decides; the LLM narrates.** Example — the dossier for
audit hit #1, Kepler-854 b:

In [7]:
from IPython.display import Markdown
dossier = (config.REPORTS_DIR / "dossiers" / "K01450.01.md")
Markdown(dossier.read_text() if dossier.exists()
         else "_Run `python -m exojury.dossier --batch` (needs FEATHERLESS_API_KEY)_")

**Vetting Dossier: K01450.01**  

**SIGNAL**  
This object is a large (59.2 Earth radii) candidate with a 2.15-day orbit, transiting its star with high signal-to-noise (334.8).  

**ASSESSMENT**  
The planet hypothesis is strongly undermined: the calibrated planet probability is 0.000, and the conformal prediction set labels it a false positive. The transit impact parameter (1.228) exceeds the physically possible maximum (~1), implying a geometry mismatch. The planet-to-star radius ratio (0.4157) is implausibly large for a planet, and the odd-even depth inconsistency (0.1422) suggests an eclipsing binary. The centroid offset (1.91σ) does not rule out contamination but is less decisive.  

**FOLLOW-UP**  
High-resolution imaging or spectroscopy to resolve the host star’s light curve and confirm/rule out a blended eclipsing binary.

## Conclusions

- **Accuracy is the least interesting number.** What matters: how much of
  it is leakage (+1.3 pts here), whether probabilities are calibrated
  (ECE 0.009), and whether the model knows when it doesn't know (26%
  abstention on the genuinely-hard frontier vs 3% on labeled data).
- Built to **distrust labels**, the pipeline found real catalog errors that
  NASA corrected years after this snapshot.
- Everything is reproducible: `README.md` has the exact commands; every
  random seed is fixed.

*Data: NASA Exoplanet Archive KOI cumulative table (DOI 10.26133/NEA4),
NASA Exoplanet Science Institute at IPAC/Caltech. Challenge: Celesta.*